In [ ]:
from pysr import PySRRegressor
from typing import Optional
import pandas as pd
import numpy as np
import random

from ReinforcementLearningEnvironment.TrainingEnvironmentVariables import CSV_PATH
from ReinforcementLearningEnvironment.helper import (make_13x13, plot_heatmap)

In [ ]:

df = pd.read_csv(CSV_PATH)
print("DF count: ", df.shape)
# df_run = df[df["run_id"] == df["run_id"].iloc[-1]]  # last run
df_run = df[df["run_id"] == "ff302c58"]  # last run
print("DF run shape:", df_run.shape)
print("DF run shape:", df_run.columns)

df.head()

In [ ]:

rank_map = {
    "2": 2, "3": 3, "4": 4, "5": 5, "6": 6,
    "7": 7, "8": 8, "9": 9, "T": 10,
    "J": 11, "Q": 12, "K": 13, "A": 14,
}

def pocket_features(pocket: str):
    # e.g. "KdKh", "4c5c"
    r1, s1 = pocket[0], pocket[1]
    r2, s2 = pocket[2], pocket[3]

    v1, v2 = rank_map[r1], rank_map[r2]

    return {
        "rank_high": max(v1, v2),
        "rank_low": min(v1, v2), 
        # "rank_gap": abs(v1 - v2),
        # "sum_ranks": v1 + v2,
        "is_pair": int(v1 == v2),
        "is_suited": int(s1 == s2),
    }

# Expand pocket into features
features = df["pocket"].apply(pocket_features).apply(pd.Series)

X = features
y = df["bb_delta"] / df["bb_delta"].abs().mean()

In [ ]:
niterations=500
binary_operators=["+", "-", "*", "/", "^", "logical_or", "logical_and"]
unary_operators=["log", "sqrt", "abs", "square", "cube", "round", "exp"]
maxsize=10
loss="loss(x, y) = (x - y)^2"

In [ ]:

# model = PySRRegressor(
#     niterations=niterations,
#     binary_operators=binary_operators,
#     unary_operators=unary_operators,
#     maxsize=maxsize,              # controls equation complexity
#     loss=loss,
# )

# model.fit(X, y)

# print(model)


In [ ]:
def gaussian_smooth_on_ranks(
    X_in: pd.DataFrame,
    y_in: pd.Series,
    sigma: float = 1.25,   # larger => smoother (try 1.0–1.5)
) -> pd.Series:
    Xr = X_in[["rank_high", "rank_low"]].to_numpy(dtype=float)
    yv = y_in.to_numpy(dtype=float)

    d2 = (
        (Xr[:, None, 0] - Xr[None, :, 0]) ** 2 +
        (Xr[:, None, 1] - Xr[None, :, 1]) ** 2
    )
    W = np.exp(-0.5 * d2 / (sigma ** 2))
    Wsum = W.sum(axis=1)
    Wsum[Wsum == 0.0] = 1.0
    return pd.Series((W @ yv) / Wsum, index=y_in.index)

# ----------------------------
# Base dataset
# ----------------------------
df = pd.read_csv(CSV_PATH)
print("DF count: ", df.shape)
# df_run = df[df["run_id"] == df["run_id"].iloc[-1]]  # last run
df_run = df[(df["run_id"] == "ff302c58") & (df["player_id"] == 3)]

print(df_run.head())

X = df_run["pocket"].apply(pocket_features).apply(pd.Series)
y = df_run["bb_delta"] / df_run["bb_delta"].abs().mean()

# ----------------------------
# 1) BINNING (average per hand-type)
# ----------------------------
tmp = X.copy()
tmp["y"] = y.values

binned = (
    tmp.groupby(["rank_high", "rank_low", "is_pair", "is_suited"], as_index=False)
       .agg(y=("y", "mean"), n=("y", "size"))
)

X_binned = binned[["rank_high", "rank_low", "is_pair", "is_suited"]]
y_binned = binned["y"]



# apply smoothing per discrete slice
y_smooth_parts = []
X_smooth_parts = []

for (suited, pair), idx in X_binned.groupby(["is_suited", "is_pair"]).groups.items():
    Xi = X_binned.loc[idx].copy()
    yi = y_binned.loc[idx].copy()
    yi_sm = gaussian_smooth_on_ranks(Xi, yi, sigma=1.25)

    X_smooth_parts.append(Xi)
    y_smooth_parts.append(yi_sm)

X_smooth = pd.concat(X_smooth_parts, axis=0).reset_index(drop=True)
y_smooth = pd.concat(y_smooth_parts, axis=0).reset_index(drop=True)

# ----------------------------
# Fit SR on SMOOTHED + BINNED
# ----------------------------
model = PySRRegressor(
    niterations=niterations,
    binary_operators=binary_operators,
    unary_operators=unary_operators,
    maxsize=maxsize,
    loss=loss,
)

model.fit(X_smooth, y_smooth)
print(model)


In [ ]:
import pandas as pd

# Load the heatmap CSV (assuming it's saved as a file or in a string)
heatmap_df = pd.read_csv('theoretical_data/preflop_4p.csv', index_col=0)

# Function to extract features from rank pair and suitedness
def heatmap_features(rank1, rank2, is_suited):
    v1 = rank_map[rank1]
    v2 = rank_map[rank2]
    return {
        "rank_high": max(v1, v2),
        "rank_low": min(v1, v2),
        "is_pair": int(v1 == v2),
        "is_suited": int(is_suited)
    }

# List to store rows
data = []

# Iterate over rank pairs
ranks = heatmap_df.columns.tolist()
for i, rank1 in enumerate(ranks):
    for j, rank2 in enumerate(ranks):
        if i < j:  # Upper triangle: suited hands
            ev = heatmap_df.iloc[i, j]
            features = heatmap_features(rank1, rank2, is_suited=1)
        elif i > j:  # Lower triangle: offsuit hands
            ev = heatmap_df.iloc[i, j]
            features = heatmap_features(rank1, rank2, is_suited=0)
        else:  # Diagonal: pairs (always suited or offsuit)
            ev = heatmap_df.iloc[i, j]
            features = heatmap_features(rank1, rank2, is_suited=1)  # pairs are identical, so suitedness doesn't matter
        
        # Add EV (target) and features to dataset
        features['ev'] = ev
        data.append(features)

# Convert to DataFrame
heatmap_features_df = pd.DataFrame(data)

# Now you have X and y from the heatmap
X_heatmap = heatmap_features_df.drop(columns=['ev'])
y_heatmap = heatmap_features_df['ev']

model_thero = PySRRegressor(
    niterations=niterations,
    binary_operators=binary_operators,
    unary_operators=unary_operators,
    maxsize=maxsize,              # controls equation complexity
    loss=loss,
)

model_thero.fit(X_heatmap, y_heatmap)

print(model_thero)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

RANKS = list("AKQJT98765432")

def plot_heatmap_from_csv(
    csv_path: str,
    title: Optional[str] = None,
    value_label: str = "Value",
    as_percent: bool = False,
):
    """
    Plot a 13x13 poker heatmap from a CSV produced by your merge step.

    CSV format:
      rows = A..2
      cols = A..2
    """

    df = pd.read_csv(csv_path, index_col=0)

    # Match poker convention:
    # Y-axis: A..2
    # X-axis: 2..A
    df_plot = df[RANKS[::-1]]

    data = df_plot.values
    if as_percent:
        data = data * 100.0

    plt.figure(figsize=(9, 7))
    plt.imshow(data, aspect="auto")
    plt.xticks(range(13), RANKS[::-1])
    plt.yticks(range(13), RANKS)
    plt.xlabel("Rank")
    plt.ylabel("Rank")

    if title:
        plt.title(title)

    plt.colorbar(label=value_label)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_heatmap_from_csv(
    "known_data/preflop_merged_4p.csv",
    title="6 Players – Preflop Equity (Suited ↑ / Offsuit ↓)",
    value_label="Win %",
    as_percent=True
)
grid_ev = make_13x13(df_run, value_col="bb_delta", agg="mean", min_samples=30)
plot_heatmap(grid_ev, "Pocket EV (mean BB/hand)")

In [ ]:

rank_map = {
    "2": 2, "3": 3, "4": 4, "5": 5, "6": 6,
    "7": 7, "8": 8, "9": 9, "T": 10,
    "J": 11, "Q": 12, "K": 13, "A": 14,
}

def pocket_features(pocket: str):
    # e.g. "KdKh", "4c5c"
    r1, s1 = pocket[0], pocket[1]
    r2, s2 = pocket[2], pocket[3]

    v1, v2 = rank_map[r1], rank_map[r2]

    return {
        "rank_high": max(v1, v2),
        "rank_low": min(v1, v2), 
        # "rank_gap": abs(v1 - v2),
        # "sum_ranks": v1 + v2,
        "is_pair": int(v1 == v2),
        "is_suited": int(s1 == s2),
    }

# Expand pocket into features
features = df["pocket"].apply(pocket_features).apply(pd.Series)

X = features
y = df["bb_delta"] / df["bb_delta"].abs().mean()